# core

Object-oriented projection-rug maps for compact multivariate visual comparison.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

The package intentionally stays small: pandas/numpy for data handling, matplotlib for drawing, and Palmer Penguins for the demo dataset.

In [ ]:
#| export
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from palmerpenguins import load_penguins as _load_penguins

## Demo Data And Layout

The demo layout is a sparse chain. Adjacent panels share one variable, so a highlighted observation can be followed through the map.

In [ ]:
#| export
def load_penguins():
    """Load and clean the Palmer Penguins dataset for the rugprint demo."""
    features = [
        "bill_length_mm",
        "bill_depth_mm",
        "flipper_length_mm",
        "body_mass_g",
    ]
    group = "species"
    data = _load_penguins()
    return data.dropna(subset=features + [group]).reset_index(drop=True)


def default_penguins_layout():
    """Return the sparse Palmer Penguins projection layout used in the demo."""
    projections = [
        ("bill_length_mm", "bill_depth_mm"),
        ("bill_length_mm", "flipper_length_mm"),
        ("body_mass_g", "flipper_length_mm"),
        ("body_mass_g", "bill_depth_mm"),
    ]
    layout = {
        ("bill_length_mm", "bill_depth_mm"): (0, 1),
        ("bill_length_mm", "flipper_length_mm"): (1, 1),
        ("body_mass_g", "flipper_length_mm"): (1, 2),
        ("body_mass_g", "bill_depth_mm"): (2, 2),
    }
    return projections, layout


def _default_layout(projections):
    anchor = [(0, 1), (1, 1), (1, 2), (2, 2), (3, 2), (2, 1), (1, 0), (0, 2)]
    return {projection: anchor[i] if i < len(anchor) else (i // 2, i % 2) for i, projection in enumerate(projections)}

## Data Preparation Helpers

These helpers keep the public class focused on the diagram idea: normalize inputs, compute reusable limits, resolve highlighted rows, and assign stable colors.

In [ ]:
#| export
def _limits_by_variable(data, projections):
    variables = sorted({variable for projection in projections for variable in projection})
    limits = {}
    for variable in variables:
        values = pd.to_numeric(data[variable], errors="coerce").dropna().to_numpy(dtype=float)
        if values.size == 0:
            raise ValueError(f"Column {variable!r} has no numeric values to plot.")
        lo = float(np.nanmin(values))
        hi = float(np.nanmax(values))
        span = hi - lo
        pad = span * 0.07 if span else max(abs(hi) * 0.07, 1.0)
        limits[variable] = (lo - pad, hi + pad)
    return limits


def _resolve_highlight(data, highlight):
    if highlight is None:
        return None
    if isinstance(highlight, (int, np.integer)) and -len(data) <= int(highlight) < len(data):
        return data.iloc[int(highlight)]
    if highlight in data.index:
        return data.loc[highlight]
    raise KeyError("highlight must be an integer row position or a dataframe index label")


def _color_values(data, group):
    if group is None:
        return None, None, np.array(["#3b5b92"] * len(data)), None
    if group not in data.columns:
        raise KeyError(f"Group column {group!r} is not in the dataframe.")
    labels = pd.Series(data[group]).astype("category")
    categories = list(labels.cat.categories)
    cmap = plt.get_cmap("tab10" if len(categories) <= 10 else "tab20")
    palette = {category: cmap(i % cmap.N) for i, category in enumerate(categories)}
    colors = [palette[label] for label in labels.to_numpy()]
    return labels, categories, colors, palette

## Rug Edge Planning

`rug_edges="shared"` places rugs on facing edges when adjacent panels share a variable. This is what turns independent scatterplots into a projection-rug map.

In [ ]:
#| export
def _axis_for_variable(projection, variable):
    if projection[0] == variable:
        return "x"
    if projection[1] == variable:
        return "y"
    return None


def _rug_edge_plan(projections, layout):
    edges = {projection: {"x": "bottom", "y": "left"} for projection in projections}
    for left, right in combinations(projections, 2):
        row_a, col_a = layout[left]
        row_b, col_b = layout[right]
        if abs(row_a - row_b) + abs(col_a - col_b) != 1:
            continue
        for variable in set(left).intersection(right):
            axis_a = _axis_for_variable(left, variable)
            axis_b = _axis_for_variable(right, variable)
            if axis_a == axis_b == "x" and col_a == col_b:
                if row_a < row_b:
                    edges[left]["x"], edges[right]["x"] = "bottom", "top"
                else:
                    edges[left]["x"], edges[right]["x"] = "top", "bottom"
            elif axis_a == axis_b == "y" and row_a == row_b:
                if col_a < col_b:
                    edges[left]["y"], edges[right]["y"] = "right", "left"
                else:
                    edges[left]["y"], edges[right]["y"] = "left", "right"
    return edges


def _outer_edge_plan(projections, layout):
    cols = [layout[projection][1] for projection in projections]
    min_col, max_col = min(cols), max(cols)
    mid_col = (min_col + max_col) / 2
    edges = {}
    for projection in projections:
        _, col = layout[projection]
        edges[projection] = {"x": "bottom", "y": "left" if col <= mid_col else "right"}
    return edges


def _normalize_rug_edges(rug_edges, projections, layout):
    if rug_edges is None or rug_edges == "minimal":
        return {projection: {"x": "bottom", "y": "left"} for projection in projections}
    if rug_edges == "shared":
        return _rug_edge_plan(projections, layout)
    if rug_edges == "outer":
        return _outer_edge_plan(projections, layout)
    if not isinstance(rug_edges, dict):
        raise ValueError("rug_edges must be None, 'minimal', 'shared', 'outer', or a dict.")

    normalized = {projection: {"x": None, "y": None} for projection in projections}
    for projection in projections:
        edge_spec = rug_edges.get(projection, rug_edges.get(tuple(projection), ()))
        if isinstance(edge_spec, str):
            edge_spec = (edge_spec,)
        for edge in edge_spec:
            if edge in {"bottom", "top"}:
                normalized[projection]["x"] = edge
            elif edge in {"left", "right"}:
                normalized[projection]["y"] = edge
            elif edge is not None:
                raise ValueError(f"Unknown rug edge {edge!r}; use top, bottom, left, or right.")
    return normalized

## Sparse Axes Geometry

Matplotlib subplots imply a rectangular grid. Rugprint uses manual axes so missing layout cells remain true empty space.

In [ ]:
#| export
def _manual_axes(fig, projections, layout, panel_size=1.0, diagram_mode=True, panel_gap=0.04):
    rows = np.array([layout[projection][0] for projection in projections], dtype=float)
    cols = np.array([layout[projection][1] for projection in projections], dtype=float)
    min_row, max_row = rows.min(), rows.max()
    min_col, max_col = cols.min(), cols.max()
    row_span = max(max_row - min_row, 1.0)
    col_span = max(max_col - min_col, 1.0)

    left_margin = 0.08 if diagram_mode else 0.10
    right_margin = 0.06
    bottom_margin = 0.08
    top_margin = 0.13 if diagram_mode else 0.17
    available_w = 1 - left_margin - right_margin
    available_h = 1 - bottom_margin - top_margin
    gutter = panel_gap if diagram_mode else max(panel_gap, 0.12)
    fig_w, fig_h = fig.get_size_inches()
    max_panel_w = available_w / (col_span + 1 + gutter * col_span) * fig_w
    max_panel_h = available_h / (row_span + 1 + gutter * row_span) * fig_h
    panel_in = min(max_panel_w, max_panel_h) * panel_size
    panel_w = panel_in / fig_w
    panel_h = panel_in / fig_h
    step_x = panel_w * (1 + gutter)
    step_y = panel_h * (1 + gutter)

    axes = {}
    for projection in projections:
        row, col = layout[projection]
        left = left_margin + (col - min_col) * step_x
        bottom = bottom_margin + (max_row - row) * step_y
        axes[projection] = fig.add_axes([left, bottom, panel_w, panel_h])
    return axes

## Drawing Primitives

Rugs live just outside panel borders. The highlighted point gets local point-to-rug guides inside each panel.

In [ ]:
#| export
def _draw_x_rug(ax, values, colors, edge, alpha, rug_length, rug_gap=0.01, linewidth=0.45):
    if edge is None:
        return None
    ylim = ax.get_ylim()
    span = ylim[1] - ylim[0]
    length = rug_length * span
    gap = rug_gap * span
    if edge == "top":
        y0, y1 = ylim[1] + gap, ylim[1] + gap + length
    else:
        y0, y1 = ylim[0] - gap - length, ylim[0] - gap
    lines = ax.vlines(values, y0, y1, colors=colors, alpha=alpha, linewidth=linewidth, zorder=2)
    lines.set_clip_on(False)
    return y0, y1


def _draw_y_rug(ax, values, colors, edge, alpha, rug_length, rug_gap=0.01, linewidth=0.45):
    if edge is None:
        return None
    xlim = ax.get_xlim()
    span = xlim[1] - xlim[0]
    length = rug_length * span
    gap = rug_gap * span
    if edge == "right":
        x0, x1 = xlim[1] + gap, xlim[1] + gap + length
    else:
        x0, x1 = xlim[0] - gap - length, xlim[0] - gap
    lines = ax.hlines(values, x0, x1, colors=colors, alpha=alpha, linewidth=linewidth, zorder=2)
    lines.set_clip_on(False)
    return x0, x1


def _add_variable_labels(ax, x, y, x_edge, y_edge, diagram_mode):
    if not diagram_mode:
        ax.set_xlabel(x.replace("_", " "), fontsize=8, labelpad=2, color="#344054")
        ax.set_ylabel(y.replace("_", " "), fontsize=8, labelpad=2, color="#344054")
        return
    label_style = dict(color="#475467", fontsize=6.7, alpha=0.9, transform=ax.transAxes, clip_on=False)
    x_y = 0.04 if x_edge != "top" else 0.96
    x_va = "bottom" if x_edge != "top" else "top"
    ax.text(0.98, x_y, x.replace("_", " "), ha="right", va=x_va, **label_style)
    y_x = 0.03 if y_edge != "right" else 0.97
    y_ha = "left" if y_edge != "right" else "right"
    ax.text(y_x, 0.50, y.replace("_", " "), rotation=90, ha=y_ha, va="center", **label_style)

## Highlight Connectors

The tracked observation should read as one continuous thread through the map. These helpers connect the highlighted coordinate through shared rug gutters.

In [ ]:
#| export
def _edge_point(ax, variable, projection, edge_plan, highlight_row):
    axis = _axis_for_variable(projection, variable)
    if axis is None:
        return None
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    value = float(highlight_row[variable])
    if axis == "x":
        edge = edge_plan[projection]["x"]
        if edge is None:
            return None
        return value, ylim[1] if edge == "top" else ylim[0]
    edge = edge_plan[projection]["y"]
    if edge is None:
        return None
    return (xlim[1] if edge == "right" else xlim[0]), value


def _figure_point(fig, ax, xy):
    display = ax.transData.transform(xy)
    return fig.transFigure.inverted().transform(display)


def _connector_linestyle(connector_style):
    if connector_style == "dotted":
        return (0, (1, 2.2))
    if connector_style == "dashed":
        return (0, (3, 3))
    return connector_style


def _draw_shared_rug_connectors(fig, axes_by_projection, projections, layout, edge_plan, highlight_row, connector_style="dotted"):
    if highlight_row is None:
        return
    fig.canvas.draw()
    linestyle = _connector_linestyle(connector_style)
    for left, right in combinations(projections, 2):
        shared = set(left).intersection(right)
        if not shared:
            continue
        row_a, col_a = layout[left]
        row_b, col_b = layout[right]
        if abs(row_a - row_b) + abs(col_a - col_b) != 1:
            continue
        for variable in shared:
            point_a = _edge_point(axes_by_projection[left], variable, left, edge_plan, highlight_row)
            point_b = _edge_point(axes_by_projection[right], variable, right, edge_plan, highlight_row)
            if point_a is None or point_b is None:
                continue
            a = _figure_point(fig, axes_by_projection[left], point_a)
            b = _figure_point(fig, axes_by_projection[right], point_b)
            line = Line2D(
                [a[0], b[0]], [a[1], b[1]], transform=fig.transFigure,
                color="#111827", linewidth=1.05, linestyle=linestyle, alpha=0.58, zorder=20,
            )
            fig.add_artist(line)

## Ranking Projection Pairs

A small helper for choosing candidate panels. It ranks feature pairs by mean distance between group centroids.

In [ ]:
#| export
def rank_pair_separation(data, features, group="species"):
    """Rank feature pairs by mean between-group centroid separation."""
    if group not in data.columns:
        raise KeyError(f"Group column {group!r} is not in the dataframe.")
    missing = [feature for feature in features if feature not in data.columns]
    if missing:
        raise KeyError(f"Feature columns are not in the dataframe: {missing}")
    clean = data.dropna(subset=list(features) + [group])
    rows = []
    for x, y in combinations(features, 2):
        centroids = clean.groupby(group, observed=True)[[x, y]].mean().dropna().to_numpy(dtype=float)
        if len(centroids) < 2:
            mean_distance = 0.0
        else:
            distances = []
            for i, j in combinations(range(len(centroids)), 2):
                distances.append(float(np.linalg.norm(centroids[i] - centroids[j])))
            mean_distance = float(np.mean(distances))
        rows.append({"x": x, "y": y, "mean_centroid_distance": mean_distance})
    return pd.DataFrame(rows).sort_values("mean_centroid_distance", ascending=False, ignore_index=True)

## The `Rugprint` Object

Like DABEST, the main constructor returns an object that stores the analysis state. The object can then draw itself, rank candidate projections, or produce a highlighted copy.

In [ ]:
#| export
class Rugprint:
    """A projection-rug map specification for a multivariate dataframe."""

    def __init__(
        self,
        data,
        projections=None,
        layout=None,
        group=None,
        highlight=None,
        figsize=(6.2, 7.0),
        panel_size=1.0,
        point_size=22,
        alpha=0.58,
        rug_alpha=0.24,
        rug_length=0.035,
        title=None,
        diagram_mode=True,
        show_axis_labels=False,
        show_tick_labels=False,
        connect_shared_rugs=True,
        connector_style="dotted",
        rug_gap=0.01,
        panel_gap=0.04,
        rug_edges="shared",
    ):
        if not isinstance(data, pd.DataFrame):
            data = pd.DataFrame(data)
        self.data = data
        self.projections = self._normalize_projections(projections)
        self.layout = self._normalize_layout(layout)
        self.group = group
        self.highlight = highlight
        self.figsize = figsize
        self.panel_size = panel_size
        self.point_size = point_size
        self.alpha = alpha
        self.rug_alpha = rug_alpha
        self.rug_length = rug_length
        self.title = title
        self.diagram_mode = diagram_mode
        self.show_axis_labels = show_axis_labels
        self.show_tick_labels = show_tick_labels
        self.connect_shared_rugs = connect_shared_rugs
        self.connector_style = connector_style
        self.rug_gap = rug_gap
        self.panel_gap = panel_gap
        self.rug_edges = rug_edges
        self._validate()

    def _normalize_projections(self, projections):
        if projections is None:
            numeric = list(self.data.select_dtypes(include=np.number).columns)
            if len(numeric) < 2:
                raise ValueError("At least two numeric columns are required when projections is None.")
            projections = list(combinations(numeric, 2))[:4]
        return [tuple(projection) for projection in projections]

    def _normalize_layout(self, layout):
        if layout is None:
            return _default_layout(self.projections)
        return {tuple(projection): tuple(position) for projection, position in layout.items()}

    def _validate(self):
        missing = sorted({column for projection in self.projections for column in projection if column not in self.data.columns})
        if missing:
            raise KeyError(f"Projection columns are not in the dataframe: {missing}")
        for projection in self.projections:
            if projection not in self.layout:
                raise KeyError(f"Missing layout coordinate for projection {projection!r}.")
        if self.group is not None and self.group not in self.data.columns:
            raise KeyError(f"Group column {self.group!r} is not in the dataframe.")

    @property
    def plot_data(self):
        """Rows with complete values for the requested projection map."""
        columns = sorted({column for pair in self.projections for column in pair})
        plot_data = self.data.dropna(subset=columns).copy()
        if self.group is not None:
            plot_data = plot_data.dropna(subset=[self.group])
        if plot_data.empty:
            raise ValueError("No complete rows are available for the requested projections.")
        return plot_data

    @property
    def highlight_row(self):
        """Resolved highlighted row, or `None` when no highlight is set."""
        return _resolve_highlight(self.plot_data, self.highlight)

    def with_highlight(self, highlight):
        """Return a copy of this Rugprint specification with a different highlighted row."""
        return Rugprint(
            self.data,
            projections=self.projections,
            layout=self.layout,
            group=self.group,
            highlight=highlight,
            figsize=self.figsize,
            panel_size=self.panel_size,
            point_size=self.point_size,
            alpha=self.alpha,
            rug_alpha=self.rug_alpha,
            rug_length=self.rug_length,
            title=self.title,
            diagram_mode=self.diagram_mode,
            show_axis_labels=self.show_axis_labels,
            show_tick_labels=self.show_tick_labels,
            connect_shared_rugs=self.connect_shared_rugs,
            connector_style=self.connector_style,
            rug_gap=self.rug_gap,
            panel_gap=self.panel_gap,
            rug_edges=self.rug_edges,
        )

    def rank_pairs(self, features=None, group=None):
        """Rank candidate projection pairs using this object's data."""
        if features is None:
            features = sorted({column for projection in self.projections for column in projection})
        if group is None:
            group = self.group
        if group is None:
            raise ValueError("group must be provided to rank projection pairs.")
        return rank_pair_separation(self.data, features, group=group)

    def plot(self, **overrides):
        """Draw the projection-rug map and return the matplotlib Figure."""
        options = {
            "figsize": self.figsize,
            "panel_size": self.panel_size,
            "point_size": self.point_size,
            "alpha": self.alpha,
            "rug_alpha": self.rug_alpha,
            "rug_length": self.rug_length,
            "title": self.title,
            "diagram_mode": self.diagram_mode,
            "show_axis_labels": self.show_axis_labels,
            "show_tick_labels": self.show_tick_labels,
            "connect_shared_rugs": self.connect_shared_rugs,
            "connector_style": self.connector_style,
            "rug_gap": self.rug_gap,
            "panel_gap": self.panel_gap,
            "rug_edges": self.rug_edges,
            "highlight": self.highlight,
        }
        options.update(overrides)
        return _plot_rugprint(
            self.plot_data,
            self.projections,
            self.layout,
            self.group,
            **options,
        )

## Plot Engine

`Rugprint.plot()` delegates to this private function. Keeping it private makes the public API object-oriented while leaving the drawing steps easy to read.

In [ ]:
#| export
def _plot_rugprint(
    plot_data,
    projections,
    layout,
    group=None,
    highlight=None,
    figsize=(6.2, 7.0),
    panel_size=1.0,
    point_size=22,
    alpha=0.58,
    rug_alpha=0.24,
    rug_length=0.035,
    title=None,
    diagram_mode=True,
    show_axis_labels=False,
    show_tick_labels=False,
    connect_shared_rugs=True,
    connector_style="dotted",
    rug_gap=0.01,
    panel_gap=0.04,
    rug_edges="shared",
):
    highlight_row = _resolve_highlight(plot_data, highlight)
    labels, categories, colors, palette = _color_values(plot_data, group)
    limits = _limits_by_variable(plot_data, projections)
    edge_plan = _normalize_rug_edges(rug_edges, projections, layout)

    fig = plt.figure(figsize=figsize)
    fig.patch.set_facecolor("white")
    axes_by_projection = _manual_axes(fig, projections, layout, panel_size=panel_size, diagram_mode=diagram_mode, panel_gap=panel_gap)

    for projection in projections:
        x, y = projection
        ax = axes_by_projection[projection]
        ax.set_facecolor("white")
        border_color = "#d7dbe3" if diagram_mode else "#c9ced6"
        border_width = 0.7 if diagram_mode else 0.9
        for spine in ax.spines.values():
            spine.set_color(border_color)
            spine.set_linewidth(border_width)
        ax.grid(False)

        x_values = pd.to_numeric(plot_data[x], errors="coerce").to_numpy(dtype=float)
        y_values = pd.to_numeric(plot_data[y], errors="coerce").to_numpy(dtype=float)
        ax.set_xlim(*limits[x])
        ax.set_ylim(*limits[y])
        x_edge = edge_plan[projection]["x"]
        y_edge = edge_plan[projection]["y"]

        if group is None:
            ax.scatter(x_values, y_values, s=point_size, c=colors, alpha=alpha, edgecolors="white", linewidths=0.25, zorder=3)
        else:
            label_values = labels.to_numpy()
            for category in categories:
                mask = label_values == category
                ax.scatter(
                    x_values[mask],
                    y_values[mask],
                    s=point_size,
                    c=[palette[category]],
                    alpha=alpha,
                    edgecolors="white",
                    linewidths=0.25,
                    label=str(category),
                    zorder=3,
                )

        x_rug = _draw_x_rug(ax, x_values, colors, x_edge, rug_alpha, rug_length, rug_gap=rug_gap)
        y_rug = _draw_y_rug(ax, y_values, colors, y_edge, rug_alpha, rug_length, rug_gap=rug_gap)

        if highlight_row is not None:
            hx = float(highlight_row[x])
            hy = float(highlight_row[y])
            if x_edge is not None:
                x_anchor = ax.get_ylim()[1] if x_edge == "top" else ax.get_ylim()[0]
                ax.vlines(hx, min(hy, x_anchor), max(hy, x_anchor), colors="#111827", linestyles="--", linewidth=0.9, alpha=0.58, zorder=4)
                ax.vlines(hx, min(x_rug), max(x_rug), colors="#111827", linewidth=1.6, zorder=5, clip_on=False)
            if y_edge is not None:
                y_anchor = ax.get_xlim()[1] if y_edge == "right" else ax.get_xlim()[0]
                ax.hlines(hy, min(hx, y_anchor), max(hx, y_anchor), colors="#111827", linestyles="--", linewidth=0.9, alpha=0.58, zorder=4)
                ax.hlines(hy, min(y_rug), max(y_rug), colors="#111827", linewidth=1.6, zorder=5, clip_on=False)
            ax.scatter([hx], [hy], s=point_size * 2.4, facecolors="none", edgecolors="#111827", linewidths=1.4, zorder=6)
            ax.scatter([hx], [hy], s=point_size * 0.55, c="#f9fafb", edgecolors="#111827", linewidths=0.6, zorder=7)

        if diagram_mode:
            ax.tick_params(
                axis="both", which="both", length=1.5 if show_tick_labels else 0,
                labelbottom=show_tick_labels, labelleft=show_tick_labels,
                labelright=False, labeltop=False, labelsize=6, colors="#98a2b3", pad=1,
            )
        else:
            ax.tick_params(axis="both", labelsize=7, colors="#667085", length=2, width=0.6, pad=1)
        if show_axis_labels:
            _add_variable_labels(ax, x, y, x_edge, y_edge, diagram_mode)
        else:
            ax.set_xlabel("")
            ax.set_ylabel("")

    if connect_shared_rugs:
        _draw_shared_rug_connectors(fig, axes_by_projection, projections, layout, edge_plan, highlight_row, connector_style=connector_style)

    if group is not None:
        handles = [
            Line2D([0], [0], marker="o", linestyle="", markersize=4.2, markerfacecolor=palette[category], markeredgecolor="white", label=str(category))
            for category in categories
        ]
        fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.965, 0.965), ncol=min(len(handles), 3), frameon=False, fontsize=7, handletextpad=0.35, columnspacing=0.8)
    if title:
        fig.suptitle(title, y=0.965, x=0.08, ha="left", fontsize=9.5 if diagram_mode else 13, fontweight="normal", color="#101828")
    return fig

## Public Constructors

`load(...)` mirrors the DABEST style: load data and options into an object, then call methods on that object. `rugprint(...)` remains as a backwards-compatible one-shot plotting wrapper.

In [ ]:
#| export
def load(data, **kwargs):
    """Create a `Rugprint` object from a dataframe and projection-map options."""
    return Rugprint(data, **kwargs)


def rugprint(data, **kwargs):
    """Draw a projection-rug map in one call and return the matplotlib Figure."""
    return Rugprint(data, **kwargs).plot()

## Smoke Checks

In [ ]:
#| hide
sample = pd.DataFrame({
    "a": [1, 2, 3, 4, 5, 6],
    "b": [2, 3, 4, 5, 6, 7],
    "c": [9, 8, 7, 6, 5, 4],
    "species": ["x", "x", "x", "y", "y", "y"],
})
ranked = rank_pair_separation(sample, ["a", "b", "c"], group="species")
assert list(ranked.columns) == ["x", "y", "mean_centroid_distance"]
assert len(ranked) == 3
rp = load(sample, projections=[("a", "b"), ("a", "c")], layout={("a", "b"): (0, 1), ("a", "c"): (1, 1)}, group="species", highlight=0, title="Smoke test")
assert isinstance(rp, Rugprint)
assert rp.highlight_row["a"] == 1
assert len(rp.rank_pairs(["a", "b", "c"])) == 3
fig = rp.plot()
assert fig.__class__.__name__ == "Figure"
assert len(fig.axes) == 2
plt.close(fig)
fig = rugprint(sample, projections=[("a", "b")], group="species", highlight=0)
assert fig.__class__.__name__ == "Figure"
plt.close(fig)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()